**Final Merge Operation**

In [0]:
#Configuration
from delta.tables import *

#Source tables
enriched_orders_table="eccom_catalog.default.enriched_orders"
customer_analytics_table= "eccom_catalog.default.customer_analytics"
product_analytics_table= "eccom_catalog.default.product_analytics"

#Target tables
orders_target_table="eccom_catalog.default.orders_target"
customers_target_table="eccom_catalog.default.customers_target"
products_target_table="eccom_catalog.default.products_target"
inventory_target_table="eccom_catalog.default.inventory_target"
shipping_target_table="eccom_catalog.default.shipping_target"
analytics_summary_table="eccom_catalog.default.analytics_summary"

print("Starting final merge operation...")

In [0]:
from pyspark.sql.functions import col,lit,current_date,count,sum,avg,countDistinct,current_timestamp
from datetime import datetime
#Read enriched data
try:
	df_enriched_orders=spark.read.table(enriched_orders_table)
	df_customer_analytics=spark.read.table(customer_analytics_table)
	df_product_analytics=spark.read.table(product_analytics_table)

	print("Successfully loaded enriched datasets")
	print(f"Enriched orders: {df_enriched_orders.count()} records")
	print(f"Customer analytics: {df_customer_analytics.count()} records")
	print(f"Product analytics: {df_product_analytics.count()} records")

except Exception as e:
    print(f"Error loading enriched datasets: {str(e)}")
    raise

In [0]:
#Merge Orders Data with SCD2 Logic
try:
	#Prepare orders data for merge	
	df_orders_merge=df_enriched_orders.select("order_id", "customer_id", "product_id", "order_date", "order_amount","currency",
	"payment_method", "shipping_address", "order_status", "created_timestamp", "processed_timestamp", "batch_id", "source_system",
	"order_profit_margin", "estimated_clv", "season", "day_of_week", "is_weekend", "time_of_day")\
	.withColumn("effective_date", current_date()).withColumn("expiry_date", lit(None).cast("date"))\
	.withColumn("is_current", lit(True))

	#Check if target table exists
	if spark.catalog.tableExists(orders_target_table):
		df_target=spark.read.table(orders_target_table).filter(col("is_current")==True)

		#Prepare match records to insert in target table  
		changed_df=df_orders_merge.alias("s").join(df_target.alias("t"),col("t.order_id")==col("s.order_id"))\
		.filter((col("t.order_amount")!=col("s.order_amount")) | (col("t.order_status")!=col("s.order_status"))).selectExpr("s.*")

		#New records to insert in target table
		new_data=df_orders_merge.join(df_target, on="order_id",how="left_anti")

		#Update in target table to record history
		target_orders=DeltaTable.forName(spark, orders_target_table)
		target_orders.alias("t").merge(changed_df.alias("s"), "t.order_id=s.order_id and t.is_current=true").whenMatchedUpdate(
		set={"expiry_date":"current_date()","is_current":"false"}
		).execute()

		result=changed_df.unionByName(new_data)
		result.write.mode("append").format("delta").saveAsTable(orders_target_table)
		
	else:
	    df_orders_merge.write.format("delta").mode("overwrite").saveAsTable(orders_target_table)
	print("Orders merge completed successfully")

except Exception as e:
	print(f"Error in performing merge operations on orders target: {str(e)}")
	raise

In [0]:
#Merge Customers data with SCD2 logic	
try:
    #Prepare customers data for merge
    df_customer_merge=df_customer_analytics.select("customer_id", "first_name", "last_name", "email", "phone",
    "date_of_birth", "registration_date", "address", "city", "state", "zip_code", "country", "customer_tier",
    "last_login", "customer_created_timestamp", "age", "age_segment", "days_since_registration", "lifecycle_stage",
    "total_orders", "total_spent", "average_order_amount", "customer_segment")\
    .withColumn("effective_date", current_date()).withColumn("expiry_date", lit(None).cast("date")).withColumn("is_current", lit(True))

    #Check if target table exists
    if spark.catalog.tableExists(customers_target_table):
        df_customer_target=spark.read.table(customers_target_table).filter(col("is_current")==True)

        #Prepare match records to insert in target table 
        df_customer_match=df_customer_merge.alias("s").join(df_customer_target.alias("t"), on="customer_id")\
        .filter((col("t.total_orders")!=col("s.total_orders")) | (col("t.total_spent")!=col("s.total_spent")) | (col("s.customer_segment")!=col("t.customer_segment"))).select("s.*")

        #New records to insert in target table
        df_customer_new=df_customer_merge.join(df_customer_target, on="customer_id", how="left_anti")

        #Update in target table to record history
        target_customers=DeltaTable.forName(spark, customers_target_table)
        target_customers.alias("t").merge(df_customer_match, (col("t.customer_id")==col("s.customer_id")) & (col("t.is_current")==True))\
        .whenMatchedUpdate(set={"expiry_date": "current_date()", "is_current":"false"}).execute()

        result=df_customer_match.unionByName(df_customer_new)
        result.write.format("delta").mode("append").saveAsTable(customers_target_table)

    else:
        df_customer_merge.write.format("delta").mode("overwrite").saveAsTable(customers_target_table)
    print("Customers merge completed successfully")	

except Exception as e:
	print(f"Error in performing merge operations on customers target: {str(e)}")
	raise

In [0]:
#Merge Products Data with SCD2 logic
try:
	#Prepare products data for merge
	df_product_merge=df_product_analytics.select("product_id", "product_name", "category", "subcategory", "brand", "price", "product_currency",
	"product_stock_quantity", "product_weight_kg", "dimensions_cm", "color", "material", "description", "launch_date", "discontinued",
	"product_created_timestamp", "product_price_segment", "product_stock_status", "days_since_launch", "product_lifecycle_stage",
	"volume_cm3", "density_kg_per_cm3", "total_orders", "total_revenue", "unique_customers", "performance_category", "product_lifecycle")\
	.withColumn("effective_date", current_date()).withColumn("expiry_date", lit(None).cast("date")).withColumn("is_current", lit(True))

	#Check if target table exists:
	if spark.catalog.tableExists(products_target_table):
		df_product=spark.read.table(products_target_table).filter(col("is_current")==True)

		#Prepare match records to insert in target table
		df_product_match=df_product_merge.alias("s").join(df_product.alias("t"), on="product_id")\
		.filter((col("t.total_orders")!=col("s.total_orders")) | (col("t.total_revenue")!=col("s.total_revenue")) | (col("t.performance_category")!=col("s.performance_category")))\
		.select("s.*")

		#New records to insert in target table
		df_product_new=df_product_merge.join(df_product, on="product_id", how="left_anti")

		#Update in target table to record history
		target_product=DeltaTable.forName(spark, products_target_table)
		target_product.alias("t").merge(df_product_match.alias("s"), (col("t.product_id")==col("s.product_id")) & (col("t.is_current")==True))\
		.whenMatchedUpdate(set={"expiry_date": "current_date()", "is_current":"false"})

		result=df_product_match.unionByName(df_product_new)
		result.write.format("delta").mode("append").saveAsTable(products_target_table)

	else:
		df_product_merge.write.format("delta").mode("overwrite").saveAsTable(products_target_table)
	print("Products merge completed successfully")
	
except Exception as e:
	print(f"Error in performing merge operations on products target: {str(e)}")
	raise

In [0]:
#Create Analytics Summary Dashboard
try:
	#Create Comprehensive analytics summary
	analytics_summary=df_enriched_orders.agg(count("order_id").alias("total_orders"), sum("order_amount").alias("total_revenue"),
	avg("order_amount").alias("avg_order_value"), countDistinct("customer_id").alias("unique_customers"), countDistinct("product_id").alias("unique_products"),
	sum("order_profit_margin").alias("total_profit"), avg("estimated_clv").alias("avg_estimated_clv"))\
	.withColumn("profit_margin_percentage", col("total_profit")*100.0/col("total_revenue")).withColumn("revenue_per_customer", col("total_revenue")/col("unique_customers"))\
	.withColumn("orders_per_customer", col("total_orders")/col("unique_customers"))\
	.withColumn("report_date", current_date()).withColumn("report_timestamp", current_timestamp())

	#Add seasonal analysis
	seasonal_analysis=df_enriched_orders.groupBy("season").agg(count("order_id").alias("total_orders"),sum("order_amount").alias("total_revenue"),
	avg("order_amount").alias("avg_seasonal_order_value"))

	#Add customer segment analysis 
	segment_analysis=df_customer_analytics.groupBy("customer_segment").agg(count("customer_id").alias("total_customers"),sum("total_spent").alias("segment_revenue"),
	avg("total_spent").alias("avg_segment_value"))

	#Add product category analysis
	category_analysis=df_product_analytics.groupBy("category").agg(count("product_id").alias("product_count"),sum("total_revenue").alias("category_revenue"),
	avg("total_revenue").alias("avg_category_revenue"))
	print("Analytics summary created successfully")

except Exception as e:
	print(f"Error in creation of analytics summary: {str(e)}")
	raise

In [0]:
#Write Analytics Summary to Table
try:
	#Write main analytics summary
	analytics_summary.write.format("delta").mode("overwrite").saveAsTable(analytics_summary_table)

	#Write seasonal analysis 
	seasonal_analysis.write.format("delta").mode("overwrite").saveAsTable("eccom_catalog.default.seasonal_analysis")

	#Write segment analysis 
	segment_analysis.write.format("delta").mode("overwrite").saveAsTable("eccom_catalog.default.segment_analysis")

	#Write category analysis 
	category_analysis.write.format("delta").mode("overwrite").saveAsTable("eccom_catalog.default.category_analysis")

	print("Analytics summary tables created successfully")

	#Get final statics
	final_status=analytics_summary.collect()[0]
	print("Final Analytics Summary: ")
	print(f"Total orders: {final_status['total_orders']}")
	print(f"Total revenue: {final_status['total_revenue']:,.2f}")
	print(f"Average order value: {final_status['avg_order_value']:,.2f}")
	print(f"Total unique customers: {final_status['unique_customers']}")
	print(f"Total unique products: {final_status['unique_products']}")
	print(f"Total profit: {final_status['total_profit']:,.2f}")
	print(f"Profit marging: {final_status['profit_margin_percentage']:.2f}%")
	print(f"Revenue per customer: {final_status['revenue_per_customer']:,.2f}")
	print(f"Total orders per customer: {final_status['orders_per_customer']:.2f}")

except Exception as e:
	print(f"Error writing analytics summary: {str(e)}")
	raise

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType
#Write in processing log table 
import json
merge_summary={
"archived_files": None,
"invalid_records": None,
"status": "SUCCESS",
"task": "final_merge_operation",
"timestamp": datetime.now().isoformat(),
"total_records": int(final_status["total_orders"])
}	

print(f"Processing summary: {json.dumps(merge_summary)}")
log_schema=StructType([
StructField("archived_files", LongType(), True),
StructField("invalid_records", LongType(), True),
StructField("status", StringType(), True),
StructField("task", StringType(), True),
StructField("timestamp", StringType(), True),
StructField("total_records", LongType(), True)
])
processing_log=spark.createDataFrame([merge_summary], schema=log_schema)
processing_log.write.format("delta").mode("append").saveAsTable("eccom_catalog.default.processing_log")

#Set validation result for downstream tasks
try:
    dbutils.jobs.taskValues.set("final_merge_status", "SUCCESS")
    dbutils.jobs.taskValues.set("total_revenue", float(final_status["total_revenue"]))
    print(f"Task values set successfully - Total revenue: {float(final_status['total_revenue'])}")

except Exception as e:
    print(f"Warning: Could not set task values (non-critical): {e}")          

print("Event-driven pipeline processing completed successfully!")